In [1]:
import glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 50)

path = glob.glob("data/public_reviews_dataset_cleaned.csv")[0]
df = pd.read_csv(path, low_memory=False)
print(path, df.shape)
print(df.dtypes)
df.head(3)

COL = { 
    "pid": "asin",
    "rid": "reviewer_id",
    "rating": "review_rating",
    "text": "review_text",
    "date": "review_date",
}
LABEL = "fake_review_product"
LEAK = ["reviewer_classified_fake", "reviewer_classified_honest",
         "reviewer_labeled_fake", "reviewer_labeled_honest"]
CAMP = "fake_review_campaign_start_date"
DEL = "review_is_removed_by_amazon"

df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")

data/public_reviews_dataset_cleaned.csv (381734, 23)
asin                                   str
review_id                              str
reviewer_id                            str
review_title                           str
review_text                            str
review_rating                        int64
review_date                            str
product_title                          str
product_url                            str
number_of_helpful                  float64
number_of_photos                   float64
photo_thumbnail_urls                   str
photo_fullsize_urls                    str
asin_url                               str
review_url                             str
reviewer_url                           str
fake_review_campaign_start_date        str
fake_review_product                   bool
reviewer_classified_fake              bool
reviewer_classified_honest            bool
reviewer_labeled_fake              float64
reviewer_labeled_honest            float64
r

In [2]:
pid, rat, txt, dt = COL["pid"], COL["rating"], COL["text"], COL["date"]

print("yorum:", len(df), " ürün:", df[pid].nunique(), " yorumcu:", df[COL["rid"]].nunique())
print("tarih aralığı:", df[dt].min(), "->", df[dt].max())

# Etiket dağılımı (yorum düzeyinde ve ürün düzeyinde)
print(df[LABEL].value_counts(dropna=False))
print(df.groupby(pid)[LABEL].first().value_counts(dropna=False))

# Etiket bir ürün içinde sabit mi? (0 çıkmalı)
chk = df.groupby(pid)[LABEL].nunique()
print("etiketi değişen ürün sayısı:", int((chk > 1).sum()))

# Puan dağılımı ve ürün başına yorum sayısı
print(df[rat].value_counts(normalize=True).sort_index().round(3))
print(df.groupby(pid).size().describe().round(1))

# Etiketle ilgili sütunlar: ne kadarı boş, kaç farklı değer var?
for c in LEAK + [CAMP, DEL]:
    print(c, "| boş oranı:", round(df[c].isna().mean(), 3), "| farklı değer:", df[c].nunique(dropna=True))

# Silinme sütunu 15 Haziran 2020'den sonra güvenilmez:
cut = pd.Timestamp("2020-06-15")
print("15 Haz 2020 sonrası yorum oranı:", round((df[dt] >= cut).mean(), 3))

yorum: 381734  ürün: 3389  yorumcu: 334342
tarih aralığı: 2000-06-12 00:00:00 -> 2021-01-19 00:00:00
fake_review_product
False    221181
True     160553
Name: count, dtype: int64
fake_review_product
False    1895
True     1494
Name: count, dtype: int64
etiketi değişen ürün sayısı: 0
review_rating
1    0.289
2    0.117
3    0.140
4    0.278
5    0.176
Name: proportion, dtype: float64
count    3389.0
mean      112.6
std       184.4
min         1.0
25%        17.0
50%        46.0
75%       127.0
max      2251.0
dtype: float64
reviewer_classified_fake | boş oranı: 0.0 | farklı değer: 2
reviewer_classified_honest | boş oranı: 0.0 | farklı değer: 2
reviewer_labeled_fake | boş oranı: 0.79 | farklı değer: 2
reviewer_labeled_honest | boş oranı: 0.79 | farklı değer: 2
fake_review_campaign_start_date | boş oranı: 0.624 | farklı değer: 206
review_is_removed_by_amazon | boş oranı: 0.226 | farklı değer: 2
15 Haz 2020 sonrası yorum oranı: 0.228


In [3]:
g = df.groupby(COL["pid"]).agg(kampanya_var=(CAMP, lambda s: s.notna().any()), etiket=(LABEL, "first"))
pd.crosstab(g["kampanya_var"], g["etiket"])

etiket,False,True
kampanya_var,,
False,1895,45
True,0,1449


In [4]:
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("results", exist_ok=True)

d = df.dropna(subset=[pid, rat, dt]).copy()
d[txt] = d[txt].fillna("").astype(str)
d[rat] = d[rat].astype(float)

# ---- PUAN özellikleri ----
g = d.groupby(pid)[rat]
R = pd.DataFrame({"r_n": g.size(), "r_mean": g.mean(), "r_std": g.std(ddof=0)})
R["r_share5"]  = (d[rat] == 5).groupby(d[pid]).mean()
R["r_share12"] = (d[rat] <= 2).groupby(d[pid]).mean()
ct = pd.crosstab(d[pid], d[rat], normalize="index")
R["r_entropy"] = -(ct * np.log2(ct.where(ct > 0, 1))).sum(axis=1)

# ---- ZAMAN özellikleri ----
d["week"] = d[dt].dt.to_period("W").dt.start_time
wk = d.groupby([pid, "week"]).size()
T = pd.DataFrame({"t_max_week": wk.groupby(level=0).max(), "t_mean_week": wk.groupby(level=0).mean()})
T["t_burst"] = T["t_max_week"] / T["t_mean_week"]
d["age"] = (d[dt] - d.groupby(pid)[dt].transform("min")).dt.days
T["t_span_days"] = d.groupby(pid)["age"].max()
early = d[d["age"] <= 30]
T["t_early_share"] = (early.groupby(pid).size() / R["r_n"]).reindex(T.index).fillna(0)
T["t_early5"] = (early[rat] == 5).groupby(early[pid]).mean().reindex(T.index)
d["rank_pct"] = d.groupby(pid)[dt].rank(pct=True, method="first")
late_mean  = d[d["rank_pct"] > 0.5].groupby(pid)[rat].mean()
early_mean = d[d["rank_pct"] <= 0.5].groupby(pid)[rat].mean()
T["t_drift"] = (late_mean - early_mean).reindex(T.index)

# ---- METİN özellikleri ----
d["len"] = d[txt].str.split().str.len()
d["excl"] = d[txt].str.count("!")
X_text = d.groupby(pid).agg(x_len_mean=("len", "mean"), x_len_std=("len", "std"), x_excl=("excl", "mean"))
norm = d[txt].str.lower().str.strip()
X_text["x_dup"] = norm.groupby(d[pid]).apply(lambda s: s.duplicated(keep=False).mean())

def mean_sim(s):
    s = s[s.str.len() > 0]
    if len(s) < 3:
        return np.nan
    s = s.sample(min(len(s), 40), random_state=0)
    try:
        M = TfidfVectorizer().fit_transform(s)
    except ValueError:
        return np.nan
    S = cosine_similarity(M)
    n = S.shape[0]
    return (S.sum() - n) / (n * (n - 1))

X_text["x_sim"] = d.groupby(pid)[txt].agg(mean_sim)

# ---- Birleştir ----
X = R.join(T).join(X_text)
y = d.groupby(pid)[LABEL].first().astype(int).loc[X.index]

GROUPS = {
    "puan":  [c for c in X.columns if c.startswith("r_")],
    "zaman": [c for c in X.columns if c.startswith("t_")],
    "metin": [c for c in X.columns if c.startswith("x_")],
}
ALL = sum(GROUPS.values(), [])
X.assign(y=y).to_csv("results/product_table.csv")
print(X.shape, y.value_counts().to_dict())
X.describe().T.round(2)

(3389, 18) {0: 1895, 1: 1494}


,count,mean,std,min,25%,50%,75%,max
r_n,3389.0,112.64,184.41,1.00,17.00,46.00,127.00,2251.00
r_mean,3389.0,3.05,0.79,1.00,2.51,2.96,3.50,5.00
r_std,3389.0,1.27,0.36,0.00,1.18,1.32,1.47,2.00
r_share5,3389.0,0.23,0.26,0.00,0.03,0.10,0.37,1.00
r_share12,3389.0,0.38,0.21,0.00,0.24,0.38,0.51,1.00
r_entropy,3389.0,1.72,0.51,-0.00,1.55,1.91,2.06,2.32
t_max_week,3389.0,6.28,8.04,1.00,2.00,4.00,7.00,123.00
t_mean_week,3389.0,2.17,2.35,1.00,1.18,1.50,2.25,50.02
t_burst,3389.0,2.54,1.14,1.00,1.79,2.47,3.14,23.23
t_span_days,3389.0,734.32,836.60,0.00,269.00,443.00,827.00,7463.00
